B31MV - Lab 2 (Week 5)
------------------------------

Bag of Visual Words (BoVW) image classification
------------------------------------------------
BoVW is a feature extraction technique used in image classification. It treats local features in an image (e.g., edges, corners) like words in a document and builds a "visual vocabulary." Images are then represented as histograms of these visual words, which can be fed into a machine learning model for classification.

Reference Video: https://www.youtube.com/watch?v=jjQetJtQDS4

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import zipfile
import tarfile
from skimage import io
from skimage.color import rgb2gray
from skimage.transform import resize
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout

In [5]:
## Extracting dataset
# Dataset: Caltech-101
url = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip"
dataset_path = "./caltech-101.zip"
dataset_extract_path = "./caltech-101"

if not os.path.exists(dataset_extract_path): # Check if dataset is already extracted
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, dataset_path) # Download the dataset

    print("Extracting dataset...")
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref: # Extract the dataset
        zip_ref.extractall("./") # Extract to current directory
    print("Dataset ready.")

In [7]:
archive_path = "./caltech-101/101_ObjectCategories.tar.gz" # Path to the tar.gz file inside the extracted folder
dataset_extract_path = "./101_ObjectCategories" # Path where the tar.gz file will be extracted

if not os.path.exists(dataset_extract_path): # Check if dataset is already extracted
    print("Extracting dataset...")
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path="./") # Extract to current directory
    print("Dataset ready.")

In [6]:
categories = ["airplanes", "Motorbikes", "Faces", "watch"] # Define the categories to be used for classification
image_size = (150, 150)  # Resize images to fixed size

In [8]:
def load_images(category): # Function to load images from a specific category
    images = []
    category_path = os.path.join(dataset_extract_path, category)
    if not os.path.exists(category_path):
        return []
    for img_name in os.listdir(category_path)[:50]:  # Load 50 images per category
        img_path = os.path.join(category_path, img_name)
        try:
            img = io.imread(img_path)
            img = rgb2gray(img)  # Convert to grayscale
            img = resize(img, image_size)  # Resize
            images.append(img)
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
    return images

In [9]:
# Load images and labels
# This code iterates through the defined categories, loads images from each category using the load_images function, 
# and stores the images and their corresponding labels in separate lists.
image_data = []
labels = []
for label, category in enumerate(categories):
    imgs = load_images(category)
    image_data.extend(imgs)
    labels.extend([label] * len(imgs))

In [10]:
# Convert to numpy array
image_data = np.array(image_data)
labels = np.array(labels)

## Section 3 — Visualise Sample Images

In [ ]:
# Section 3: Visualise sample images from each category
fig, axes = plt.subplots(len(categories), 5, figsize=(15, 3 * len(categories)))

for row, category in enumerate(categories):
    cat_path = os.path.join(dataset_extract_path, category)
    img_files = [f for f in os.listdir(cat_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:5]
    for col in range(5):
        ax = axes[row, col]
        if col < len(img_files):
            img = io.imread(os.path.join(cat_path, img_files[col]))
            ax.imshow(img, cmap='gray')
            ax.set_title(category if col == 0 else '', fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample Images — Caltech-101 (4 categories)', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Total images loaded : {len(image_data)}")
for i, cat in enumerate(categories):
    print(f"  {cat:<14} : {(labels == i).sum()} images")


## Section 4 — SIFT Feature Extraction

In [ ]:
# Section 4: Extract SIFT descriptors from every image.
# SIFT produces a variable number of 128-dim keypoint descriptors per image.
# We collect all descriptors across the training set to build the vocabulary.

sift = cv2.SIFT_create()

def extract_sift_descriptors(img_float):
    """Convert float [0,1] image to uint8 and run SIFT."""
    img_uint8 = (img_float * 255).astype(np.uint8)
    _, descriptors = sift.detectAndCompute(img_uint8, None)
    return descriptors  # None if no keypoints found

print("Extracting SIFT descriptors...")
all_descriptors   = []   # pooled across all images — used to build vocabulary
image_descriptors = []   # per-image — used for BoVW encoding

for i, img in enumerate(image_data):
    desc = extract_sift_descriptors(img)
    if desc is not None:
        all_descriptors.append(desc)
        image_descriptors.append(desc)
    else:
        image_descriptors.append(np.zeros((1, 128), dtype=np.float32))
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(image_data)} images processed")

all_descriptors = np.vstack(all_descriptors)
print(f"\nTotal SIFT descriptors : {len(all_descriptors):,}")
print(f"Descriptor dimension   : {all_descriptors.shape[1]}")


## Section 5 — Build Visual Vocabulary (K-Means)

In [ ]:
# Section 5: Cluster all SIFT descriptors into K visual words.
# Each cluster centre becomes a prototype local pattern — a "visual word".
# K is the vocabulary size; larger K = finer vocabulary, higher cost.

K = 100  # vocabulary size

print(f"Building visual vocabulary (K={K} words)...")
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(all_descriptors)
print(f"Vocabulary built — {K} visual words from {len(all_descriptors):,} descriptors.")


## Section 6 — BoVW Histogram Encoding

In [ ]:
# Section 6: Encode each image as a normalised frequency histogram over the vocabulary.
# Each bin counts how often each visual word appears in the image.
# The result is a fixed-length K-dimensional vector regardless of image content.

def encode_bovw(descriptors, kmeans_model, k):
    histogram = np.zeros(k, dtype=np.float32)
    if descriptors is not None and len(descriptors) > 0:
        word_assignments = kmeans_model.predict(descriptors)
        for w in word_assignments:
            histogram[w] += 1.0
        histogram /= (histogram.sum() + 1e-7)   # L1 normalise
    return histogram

print("Encoding images as BoVW histograms...")
X = np.array([encode_bovw(desc, kmeans, K) for desc in image_descriptors])
y = labels.copy()

print(f"Feature matrix : {X.shape}  ({X.shape[0]} images × {X.shape[1]} visual words)")

# Train / test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train split    : {X_train.shape[0]} samples")
print(f"Test split     : {X_test.shape[0]}  samples")


## Section 7 — Multi-Classifier Comparison

In [ ]:
# Section 7: Train five classical classifiers on the BoVW histograms and compare.

classifiers = {
    'SVM':                 SVC(kernel='rbf', C=10, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'MLP':                 MLPClassifier(hidden_layer_sizes=(256, 128),
                                         max_iter=500, random_state=42),
}

results = {}
for name, clf in classifiers.items():
    print(f"Training {name}...", end=' ')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    results[name] = {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1':        f1_score(y_test, y_pred, average='macro', zero_division=0),
        'y_pred':    y_pred,
    }
    print(f"Accuracy: {results[name]['accuracy']*100:.1f}%")

print("\nAll classifiers trained.")


In [ ]:
# Results bar chart — accuracy, precision, recall, F1 for every classifier
names  = list(results.keys())
accs   = [results[n]['accuracy']  * 100 for n in names]
precs  = [results[n]['precision'] * 100 for n in names]
recs   = [results[n]['recall']    * 100 for n in names]
f1s    = [results[n]['f1']        * 100 for n in names]

x, w = np.arange(len(names)), 0.2
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - 1.5*w, accs,  w, label='Accuracy',  color='steelblue',    alpha=0.85)
ax.bar(x - 0.5*w, precs, w, label='Precision', color='darkorange',   alpha=0.85)
ax.bar(x + 0.5*w, recs,  w, label='Recall',    color='seagreen',     alpha=0.85)
ax.bar(x + 1.5*w, f1s,   w, label='F1 Score',  color='mediumpurple', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Score (%)')
ax.set_title('Classifier Comparison — BoVW on Caltech-101 (4 categories)')
ax.legend()
ax.set_ylim(0, 100)
ax.axhline(25, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Random baseline')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Numeric summary table
print(f"{'Classifier':<24} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 64)
for n in names:
    r = results[n]
    print(f"{n:<24} {r['accuracy']*100:>8.1f}%"
          f" {r['precision']*100:>9.1f}%"
          f" {r['recall']*100:>7.1f}%"
          f" {r['f1']*100:>7.1f}%")


## Section 8 — Confusion Matrices

In [ ]:
# Section 8: Confusion matrix for each classifier side by side.
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, len(classifiers), figsize=(20, 4))
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=categories)
    disp.plot(ax=ax, colorbar=False, xticks_rotation=30, cmap='Blues')
    ax.set_title(name, fontsize=10, fontweight='bold')
plt.suptitle('Confusion Matrices — BoVW Classifiers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 9 — CNN Comparison (Direct Pixel Learning)

In [ ]:
# Section 9: Train a small CNN directly on pixel values as a comparison point.
# BoVW discards spatial information; CNN retains it — the accuracy gap shows why
# spatial feature hierarchies matter for image classification.

from tensorflow.keras.utils import to_categorical

X_cnn = image_data.reshape(-1, 150, 150, 1).astype(np.float32)
y_cat = to_categorical(y, num_classes=len(categories))

X_cnn_tr, X_cnn_te, y_cnn_tr, y_cnn_te = train_test_split(
    X_cnn, y_cat, test_size=0.2, random_state=42, stratify=y)

cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 1)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(len(categories), activation='softmax'),
])
cnn_model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

history = cnn_model.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=15, batch_size=16,
    validation_data=(X_cnn_te, y_cnn_te),
    verbose=1)

_, cnn_acc = cnn_model.evaluate(X_cnn_te, y_cnn_te, verbose=0)
print(f"\nCNN Test Accuracy: {cnn_acc*100:.2f}%")


In [ ]:
# CNN training history + final summary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='Train', color='steelblue')
ax1.plot(history.history['val_accuracy'], label='Val',   color='darkorange')
ax1.set_title('CNN Accuracy'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='Train', color='steelblue')
ax2.plot(history.history['val_loss'], label='Val',   color='darkorange')
ax2.set_title('CNN Loss'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('CNN Training History', fontsize=12)
plt.tight_layout()
plt.show()

best_bovw_name = names[int(np.argmax(accs))]
print(f"\n{'Model':<26} {'Accuracy':>9}")
print("-" * 38)
for n in names:
    print(f"{n:<26} {results[n]['accuracy']*100:>8.1f}%")
print(f"{'CNN (direct pixels)':<26} {cnn_acc*100:>8.1f}%")
print(f"\nBest BoVW classifier: {best_bovw_name} ({max(accs):.1f}%)")
print(f"CNN:                  {cnn_acc*100:.1f}%")
